In [ ]:
# ============================================================
# ANALYSE UNIQUE : CSV -> tendances sentiment & thèmes
# ============================================================

import pandas as pd
import re

# ------------------------------------------------------------
# 1) LOAD DATA
# ------------------------------------------------------------
CSV_PATH = "df_final.csv"
df = pd.read_csv(CSV_PATH)

# dates -> mois
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")
df["month"] = df["pubtime"].dt.to_period("M").astype(str)

# ------------------------------------------------------------
# 2) ADMINISTRATIVE REGISTER (bureaucratie vs administration)
# ------------------------------------------------------------
BUREAUCRACY_TERMS = ["bureaucratie", "bürokratie"]
ADMIN_TERMS = ["administration publique", "verwaltung"]

def classify_register(sentence: str):
    if not isinstance(sentence, str):
        return "unknown"
    s = sentence.lower()
    has_bureau = any(t in s for t in BUREAUCRACY_TERMS)
    has_admin  = any(t in s for t in ADMIN_TERMS)
    if has_bureau and has_admin:
        return "mixed"
    elif has_bureau:
        return "bureaucracy"
    elif has_admin:
        return "public_administration"
    else:
        return "unknown"

df["admin_register"] = df["sentence"].apply(classify_register)

# garder seulement les registres pertinents
df = df[df["admin_register"].isin(["bureaucracy", "public_administration"])]

# ------------------------------------------------------------
# 3) SENTIMENT TRENDS BY MONTH × REGISTER
# ------------------------------------------------------------
sentiment_month = (
    df
    .groupby(["month", "admin_register", "sentiment_label"])
    .size()
    .reset_index(name="n")
    .assign(
        pct=lambda x: x["n"]
        / x.groupby(["month", "admin_register"])["n"].transform("sum")
    )
)

# table large pour graphes (stacked area / line)
sentiment_month_wide = (
    sentiment_month
    .pivot_table(
        index="month",
        columns=["admin_register", "sentiment_label"],
        values="pct",
        fill_value=0
    )
    .sort_index()
)

# ------------------------------------------------------------
# 4) THEMES BY MONTH × REGISTER × SENTIMENT
# ------------------------------------------------------------
themes_month = (
    df
    .groupby(["month", "admin_register", "sentiment_label", "main_theme"])
    .size()
    .reset_index(name="n")
    .assign(
        pct=lambda x: x["n"]
        / x.groupby(["month", "admin_register", "sentiment_label"])["n"].transform("sum")
    )
)

# exemples prêts à l'emploi :
bureau_neg_themes = (
    themes_month
    .query("admin_register == 'bureaucracy' and sentiment_label == 'negative'")
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

admin_pos_themes = (
    themes_month
    .query("admin_register == 'public_administration' and sentiment_label == 'positive'")
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

# ------------------------------------------------------------
# 5) OUTPUTS DISPONIBLES POUR GRAPHES / INTERPRÉTATION
# ------------------------------------------------------------
print("=== Tables disponibles ===")
print("1) sentiment_month_wide  -> tendances mensuelles sentiment × registre")
print("2) bureau_neg_themes     -> thèmes dominants (bureaucratie, négatif)")
print("3) admin_pos_themes      -> thèmes dominants (administration, positif)")
print()
print("Aperçu sentiment_month_wide:")
display(sentiment_month_wide.head())


In [ ]:
import matplotlib.pyplot as plt

# --- sélectionner admin publique ---
admin_sent = sentiment_month_wide["public_administration"]

admin_sent.plot(
    kind="line",
    marker="o",
    figsize=(10, 5)
)

plt.title("Public Administration – Sentiment Trends Over Time")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.ylim(0, 1)
plt.legend(title="Sentiment")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- sélectionner bureaucratie ---
bureau_sent = sentiment_month_wide["bureaucracy"]

bureau_sent.plot(
    kind="line",
    marker="o",
    figsize=(10, 5)
)

plt.title("Bureaucracy – Sentiment Trends Over Time")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.ylim(0, 1)
plt.legend(title="Sentiment")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


admin_theme_month = (
    themes_month
    .query("admin_register == 'public_administration'")
    .groupby(["month", "main_theme"])["n"]
    .sum()
    .reset_index()
    .assign(
        pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

admin_theme_month.plot(
    kind="area",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Public Administration – Thematic Composition by Month")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.legend(title="Theme", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


bureau_theme_month = (
    themes_month
    .query("admin_register == 'bureaucracy'")
    .groupby(["month", "main_theme"])["n"]
    .sum()
    .reset_index()
    .assign(
        pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

bureau_theme_month.plot(
    kind="area",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Bureaucracy – Thematic Composition by Month")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.legend(title="Theme", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
neg_theme_month = (
    themes_month
    .query("sentiment_label == 'negative'")
    .groupby(["month", "main_theme"])["n"]
    .sum()
    .reset_index()
    .assign(
        pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

neg_theme_month.plot(
    kind="area",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Negative Sentiment – Thematic Composition by Month")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.legend(title="Theme", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
neu_theme_month = (
    themes_month
    .query("sentiment_label == 'neutral'")
    .groupby(["month", "main_theme"])["n"]
    .sum()
    .reset_index()
    .assign(
        pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

neu_theme_month.plot(
    kind="area",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Neutral Sentiment – Thematic Composition by Month")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.legend(title="Theme", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
pos_theme_month = (
    themes_month
    .query("sentiment_label == 'positive'")
    .groupby(["month", "main_theme"])["n"]
    .sum()
    .reset_index()
    .assign(
        pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )
    .pivot(index="month", columns="main_theme", values="pct")
    .fillna(0)
    .sort_index()
)

pos_theme_month.plot(
    kind="area",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Positive Sentiment – Thematic Composition by Month")
plt.ylabel("Share of sentences")
plt.xlabel("Month")
plt.legend(title="Theme", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_sentiment_with_themes(df, register, sentiment, top_k=8):
    """
    2 panneaux:
      - en haut: les 3 courbes (neg/neu/pos) pour le registre
      - en bas: aire empilée des thèmes, MAIS redimensionnée pour que le total = courbe du sentiment choisi
    """
    d = df[df["admin_register"] == register].copy()
    d = d.dropna(subset=["month", "sentiment_label", "main_theme"])

    # --- 1) Parts de sentiment par mois (dans le registre) ---
    sent = (
        d.groupby(["month", "sentiment_label"]).size().reset_index(name="n")
         .assign(pct=lambda x: x["n"] / x.groupby("month")["n"].transform("sum"))
         .pivot(index="month", columns="sentiment_label", values="pct")
         .fillna(0)
         .sort_index()
    )

    # --- 2) Composition par thèmes à l'intérieur de (sentiment) ---
    theme_in_sent = (
        d[d["sentiment_label"] == sentiment]
        .groupby(["month", "main_theme"]).size().reset_index(name="n")
    )

    if theme_in_sent.empty:
        print(f"⚠️ Aucun cas pour register={register} / sentiment={sentiment}")
        return

    theme_in_sent = theme_in_sent.assign(
        pct_in_sent=lambda x: x["n"] / x.groupby("month")["n"].transform("sum")
    )

    theme_wide = (
        theme_in_sent.pivot(index="month", columns="main_theme", values="pct_in_sent")
        .fillna(0)
        .sort_index()
    )

    # --- 3) Top K thèmes + "Other" pour lisibilité ---
    top_themes = theme_wide.sum(axis=0).sort_values(ascending=False).head(top_k).index
    theme_top = theme_wide[top_themes].copy()
    theme_top["Other"] = theme_wide.drop(columns=top_themes, errors="ignore").sum(axis=1)

    # --- 4) Redimensionner pour matcher la courbe du sentiment ---
    # (sum des thèmes = part du sentiment ce mois-là)
    sentiment_curve = sent.get(sentiment)
    if sentiment_curve is None:
        print(f"⚠️ Sentiment '{sentiment}' absent des données pour register={register}")
        return

    # aligner index mois
    theme_top = theme_top.reindex(sent.index).fillna(0)
    scaled_theme = theme_top.mul(sentiment_curve, axis=0)

    # --- 5) Plot (2 panneaux) ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, height_ratios=[1, 1.4])

    # Haut: courbes sentiment
    sent.plot(kind="line", marker="o", ax=ax1)
    ax1.set_title(f"{register} — Sentiment trends (neg/neu/pos)")
    ax1.set_ylabel("Share of sentences")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.25)

    # Bas: thèmes redimensionnés = sentiment choisi
    scaled_theme.plot(kind="area", stacked=True, ax=ax2)
    ax2.plot(sentiment_curve.index, sentiment_curve.values, linewidth=2)  # overlay: la courbe du sentiment
    ax2.set_title(f"{register} — Theme composition within '{sentiment}' (scaled to match sentiment share)")
    ax2.set_ylabel("Share of ALL sentences")
    ax2.set_xlabel("Month")
    ax2.set_ylim(0, max(0.05, sentiment_curve.max() * 1.15))
    ax2.grid(True, alpha=0.25)
    ax2.legend(title="Theme", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_sentiment_with_themes(df, register="bureaucracy", sentiment="neutral", top_k=8)
plot_sentiment_with_themes(df, register="public_administration", sentiment="neutral", top_k=8)

plot_sentiment_with_themes(df, "bureaucracy", "negative", top_k=8)
plot_sentiment_with_themes(df, "bureaucracy", "positive", top_k=8)

plot_sentiment_with_themes(df, "public_administration", "negative", top_k=8)
plot_sentiment_with_themes(df, "public_administration", "positive", top_k=8)



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_sentiment_with_themes_lang(
    df,
    register,
    sentiment,
    language,
    top_k=8
):
    """
    Two-panel plot:
      - Top: sentiment trends over time (neg/neu/pos)
      - Bottom: theme composition within ONE sentiment,
        scaled so that stacked areas sum to the sentiment curve
    Conditioned on:
      - admin_register (bureaucracy / public_administration)
      - language (fr / de)
    """

    # --------------------------------------------------
    # 0) Filter data
    # --------------------------------------------------
    d = df[
        (df["admin_register"] == register) &
        (df["language"].str.lower() == language.lower())
    ].copy()

    if d.empty:
        print(f"⚠️ No data for register={register}, language={language}")
        return

    d = d.dropna(subset=["month", "sentiment_label", "main_theme"])

    # --------------------------------------------------
    # 1) Sentiment shares by month
    # --------------------------------------------------
    sent = (
        d.groupby(["month", "sentiment_label"])
        .size()
        .reset_index(name="n")
        .assign(
            pct=lambda x: x["n"]
            / x.groupby("month")["n"].transform("sum")
        )
        .pivot(index="month", columns="sentiment_label", values="pct")
        .fillna(0)
        .sort_index()
    )

    if sentiment not in sent.columns:
        print(f"⚠️ Sentiment '{sentiment}' absent for this subset.")
        return

    # --------------------------------------------------
    # 2) Theme composition within the chosen sentiment
    # --------------------------------------------------
    theme_in_sent = (
        d[d["sentiment_label"] == sentiment]
        .groupby(["month", "main_theme"])
        .size()
        .reset_index(name="n")
    )

    if theme_in_sent.empty:
        print(f"⚠️ No theme data for sentiment={sentiment}")
        return

    theme_in_sent = theme_in_sent.assign(
        pct_in_sent=lambda x: x["n"]
        / x.groupby("month")["n"].transform("sum")
    )

    theme_wide = (
        theme_in_sent
        .pivot(index="month", columns="main_theme", values="pct_in_sent")
        .fillna(0)
        .sort_index()
    )

    # --------------------------------------------------
    # 3) Top-K themes + Other
    # --------------------------------------------------
    top_themes = (
        theme_wide.sum(axis=0)
        .sort_values(ascending=False)
        .head(top_k)
        .index
    )

    theme_top = theme_wide[top_themes].copy()
    theme_top["Other"] = theme_wide.drop(columns=top_themes, errors="ignore").sum(axis=1)

    # --------------------------------------------------
    # 4) Scale themes to match sentiment curve
    # --------------------------------------------------
    sentiment_curve = sent[sentiment]
    theme_top = theme_top.reindex(sent.index).fillna(0)
    scaled_theme = theme_top.mul(sentiment_curve, axis=0)

    # --------------------------------------------------
    # 5) Plot
    # --------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(12, 8),
        sharex=True,
        height_ratios=[1, 1.4]
    )

    # Top: sentiment curves
    sent.plot(ax=ax1, marker="o")
    ax1.set_title(
        f"{register} ({language.upper()}) — Sentiment trends"
    )
    ax1.set_ylabel("Share of sentences")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.25)

    # Bottom: scaled theme composition
    scaled_theme.plot(kind="area", stacked=True, ax=ax2)
    ax2.plot(
        sentiment_curve.index,
        sentiment_curve.values,
        linewidth=2
    )

    ax2.set_title(
        f"{register} ({language.upper()}) — Theme composition within '{sentiment}'"
    )
    ax2.set_ylabel("Share of ALL sentences")
    ax2.set_xlabel("Month")
    ax2.set_ylim(0, max(0.05, sentiment_curve.max() * 1.15))
    ax2.grid(True, alpha=0.25)
    ax2.legend(
        title="Theme",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_sentiment_with_themes_lang(df, "bureaucracy", "negative", "fr")
plot_sentiment_with_themes_lang(df, "bureaucracy", "negative", "de")
plot_sentiment_with_themes_lang(df, "bureaucracy", "positive", "fr")
plot_sentiment_with_themes_lang(df, "bureaucracy", "positive", "de")
plot_sentiment_with_themes_lang(df, "bureaucracy", "neutral", "fr")
plot_sentiment_with_themes_lang(df, "bureaucracy", "neutral", "de")

plot_sentiment_with_themes_lang(df, "public_administration", "negative", "fr")
plot_sentiment_with_themes_lang(df, "public_administration", "negative", "de")
plot_sentiment_with_themes_lang(df, "public_administration", "positive", "fr")
plot_sentiment_with_themes_lang(df, "public_administration", "positive", "de")
plot_sentiment_with_themes_lang(df, "public_administration", "neutral", "fr")
plot_sentiment_with_themes_lang(df, "public_administration", "neutral", "de")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_language_share_scaled_by_sentiment(
    df,
    register,
    sentiment
):
    """
    Stacked area plot (FR vs DE) where:
    - total height = share of the sentiment in that month (within the register)
    - internal composition = FR vs DE
    """

    # ----------------------------
    # 1) Filter data
    # ----------------------------
    d = df[
        (df["admin_register"] == register) &
        (df["sentiment_label"] == sentiment)
    ].copy()

    if d.empty:
        print(f"⚠️ No data for {register} / {sentiment}")
        return

    d["language"] = d["language"].str.lower()

    # ----------------------------
    # 2) Sentiment weight by month (within register)
    # ----------------------------
    sentiment_weight = (
        df[df["admin_register"] == register]
        .groupby(["month", "sentiment_label"])
        .size()
        .reset_index(name="n")
        .assign(
            pct=lambda x: x["n"]
            / x.groupby("month")["n"].transform("sum")
        )
    )

    sentiment_weight = (
        sentiment_weight
        .query("sentiment_label == @sentiment")
        .set_index("month")["pct"]
        .sort_index()
    )

    # ----------------------------
    # 3) Language composition within sentiment
    # ----------------------------
    lang_comp = (
        d.groupby(["month", "language"])
        .size()
        .reset_index(name="n")
        .assign(
            pct_lang=lambda x: x["n"]
            / x.groupby("month")["n"].transform("sum")
        )
        .pivot(index="month", columns="language", values="pct_lang")
        .fillna(0)
        .sort_index()
    )

    # align months
    lang_comp = lang_comp.reindex(sentiment_weight.index).fillna(0)

    # ----------------------------
    # 4) Scale by sentiment weight
    # ----------------------------
    scaled_lang = lang_comp.mul(sentiment_weight, axis=0)

    # ----------------------------
    # 5) Plot
    # ----------------------------
    scaled_lang.plot(
        kind="area",
        stacked=True,
        figsize=(10, 5)
    )

    plt.title(
        f"{register.replace('_', ' ').title()} – "
        f"{sentiment.title()} sentiment\n"
        "FR vs DE (scaled by sentiment weight)"
    )
    plt.ylabel("Share of all sentences")
    plt.xlabel("Month")
    plt.ylim(0, max(0.05, sentiment_weight.max() * 1.15))
    plt.legend(title="Language")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_language_share_scaled_by_sentiment(df, "public_administration", "negative")
plot_language_share_scaled_by_sentiment(df, "bureaucracy", "negative")
plot_language_share_scaled_by_sentiment(df, "public_administration", "neutral")
plot_language_share_scaled_by_sentiment(df, "bureaucracy", "neutral")
plot_language_share_scaled_by_sentiment(df, "public_administration", "positive")
plot_language_share_scaled_by_sentiment(df, "bureaucracy", "positive")


In [ ]:
def plot_negative_share_within_language(df, register):
    """
    Line plot:
    - y = share of negative sentences WITHIN FR and WITHIN DE
    - denominator = total sentences of that language (per month, per register)
    """

    d = df[df["admin_register"] == register].copy()

    if d.empty:
        print(f"⚠️ No data for register={register}")
        return

    # total sentences per month × language
    total_lang = (
        d.groupby(["month", "language"])
        .size()
        .reset_index(name="total")
    )

    # negative sentences per month × language
    neg_lang = (
        d[d["sentiment_label"] == "negative"]
        .groupby(["month", "language"])
        .size()
        .reset_index(name="neg")
    )

    # merge + compute share
    neg_share = (
        total_lang
        .merge(neg_lang, on=["month", "language"], how="left")
        .fillna({"neg": 0})
        .assign(neg_share=lambda x: x["neg"] / x["total"])
        .pivot(index="month", columns="language", values="neg_share")
        .fillna(0)
        .sort_index()
    )

    # plot
    neg_share.plot(
        kind="line",
        marker="o",
        figsize=(10, 5)
    )

    plt.title(
        f"{register.replace('_', ' ').title()} — "
        "Share of negative sentiment within FR and DE"
    )
    plt.ylabel("Share of negative sentences (within language)")
    plt.xlabel("Month")
    plt.ylim(0, neg_share.max().max() * 1.2)
    plt.legend(title="Language")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_negative_share_within_language(df, "bureaucracy")
plot_negative_share_within_language(df, "public_administration")
